## ADMET & Drug-Likeness Evaluation (Lipinski's Rule of 5)
OverviewHigh binding affinity alone does not guarantee a viable drug candidate; a molecule must also possess favorable pharmacokinetic properties. This notebook automates the in silico evaluation of ADMET (Absorption, Distribution, Metabolism, Excretion, and Toxicity) profiles for all docked ligands. By calculating key physicochemical properties, this script filters the ligands based on Lipinski's Rule of Five to determine their oral bioavailability and "drug-likeness."
## Workflow & Methodology

### Google Drive Integration:
Mounts the workspace to access the initial 3D ligand structures generated in Phase 1.
### Cheminformatics Parsing:
Utilizes RDKit to parse the 3D .pdb ligand structures and compute exact molecular descriptors.
### Property Calculation:
Automatically calculates the following parameters for each ligand:
Molecular Weight (MW): Ideal is < 500 Daltons.
### Lipophilicity (LogP):
Ideal is < 5 (ensures cell membrane permeability).
-Hydrogen Bond Donors (HBD): Ideal is < 5.
-Hydrogen Bond Acceptors (HBA): Ideal is < 10.
-Topological Polar Surface Area (TPSA): Predicts drug transport properties (e.g., blood-brain barrier penetration).


##Drug-Likeness Validation:
Evaluates the calculated properties against Lipinski's parameters. A molecule is flagged as "Drug-Like" if it has 1 or zero violations.
##Data Export:
Aggregates all calculations into a structured Pandas DataFrame and exports it as a publication-ready .csv file.

##Dependencies

Python Libraries:
 rdkit (Cheminformatics computations),
 pandas (Dataframe management),
 os, glob, google.colab.drive.

In [2]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 28.0 MB/s eta 0:00:00


In [3]:
import os
import glob
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
from google.colab import drive

# --- 1. Mount Google Drive ---
print("Connecting to Google Drive...")
drive.mount('/content/drive', force_remount=True)

# --- 2. Define Paths ---
BASE_DIR = "/content/drive/MyDrive/Docking_Pipeline"
RAW_DIR = os.path.join(BASE_DIR, "raw_data")
ADMET_DIR = os.path.join(BASE_DIR, "admet_results")
os.makedirs(ADMET_DIR, exist_ok=True)

print("\n--- Starting ADMET / Drug-Likeness Analysis ---")

# --- 3. Calculate Lipinski Properties ---
ligand_files = glob.glob(os.path.join(RAW_DIR, "ligand_*_3D.pdb"))
admet_data = []

if not ligand_files:
    print("❌ No ligand files found in the raw_data folder! Check your Drive path.")
else:
    for lig_pdb in ligand_files:
        ligand_name = os.path.basename(lig_pdb).replace("_3D.pdb", "")

        # Load molecule in RDKit
        mol = Chem.MolFromPDBFile(lig_pdb)

        if mol:
            # Calculate Properties
            mol_wt = Descriptors.MolWt(mol)           # Molecular Weight (<500 Da)
            logp = Descriptors.MolLogP(mol)           # Lipophilicity (<5)
            h_donors = Descriptors.NumHDonors(mol)    # Hydrogen Bond Donors (<5)
            h_acceptors = Descriptors.NumHAcceptors(mol)# Hydrogen Bond Acceptors (<10)
            tpsa = Descriptors.TPSA(mol)              # Topological Polar Surface Area

            # Check Lipinski Violations
            violations = sum([mol_wt > 500, logp > 5, h_donors > 5, h_acceptors > 10])

            admet_data.append({
                "Ligand": ligand_name,
                "MW (Da)": round(mol_wt, 2),
                "LogP": round(logp, 2),
                "H-Donors": h_donors,
                "H-Acceptors": h_acceptors,
                "TPSA": round(tpsa, 2),
                "Lipinski_Violations": violations,
                "Drug-Like": "Yes" if violations <= 1 else "No"
            })

    # --- 4. Save to CSV ---
    if admet_data:
        df_admet = pd.DataFrame(admet_data)
        csv_path = os.path.join(ADMET_DIR, "admet_summary.csv")
        df_admet.to_csv(csv_path, index=False)
        print(f"✅ ADMET data successfully saved to: {csv_path}\n")
        print(df_admet)
    else:
        print("❌ Could not process ligands for ADMET.")

Connecting to Google Drive...
Mounted at /content/drive

--- Starting ADMET / Drug-Likeness Analysis ---
✅ ADMET data successfully saved to: /content/drive/MyDrive/Docking_Pipeline/admet_results/admet_summary.csv

           Ligand  MW (Da)  LogP  H-Donors  H-Acceptors   TPSA  \
0  ligand_5281034   332.48  4.40         2            3  57.53   
1     ligand_2244   180.16  1.31         1            3  63.60   
2     ligand_3672   206.28  3.07         1            1  37.30   
3     ligand_5291   493.62  4.59         2            7  86.28   
4   ligand_123631   446.91  4.28         1            7  68.74   

   Lipinski_Violations Drug-Like  
0                    0       Yes  
1                    0       Yes  
2                    0       Yes  
3                    0       Yes  
4                    0       Yes  
